In [0]:
#IMPORT

from pyspark.sql import functions as F
from pyspark.sql.window import Window

purchase_orders = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/purchase_orders"
)

vendors = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/vendors"
)

vendor_invoices = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/vendor_invoices"
)

hsn_rate_schedule = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/hsn_rate_schedule"
)

cbic_blacklist = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/cbic_blacklist"
)

historical_po_values = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/historical_po_values"
)

ground_truth = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Bronze/ground_truth"
)

In [0]:
display(purchase_orders.printSchema())
display(vendors.printSchema())
display(vendor_invoices.printSchema())
display(hsn_rate_schedule.printSchema())
display(cbic_blacklist.printSchema())
display(historical_po_values.printSchema())
display(ground_truth.printSchema())

root
 |-- po_id: string (nullable = true)
 |-- po_date: date (nullable = true)
 |-- buyer_gstin: string (nullable = true)
 |-- buyer_state: string (nullable = true)
 |-- hsn_code: string (nullable = true)
 |-- product_desc: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit: string (nullable = true)
 |-- base_amount: double (nullable = true)
 |-- cgst_rate: double (nullable = true)
 |-- sgst_rate: double (nullable = true)
 |-- igst_rate: double (nullable = true)
 |-- cgst_amt: double (nullable = true)
 |-- sgst_amt: double (nullable = true)
 |-- igst_amt: double (nullable = true)
 |-- cess_amt: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- ewb_number: string (nullable = true)
 |-- ewb_generated_date: string (nullable = true)
 |-- place_of_supply: string (nullable = true)
 |-- delivery_state: string (nullable = true)
 |-- invoice_billing_name: string (nullable = true)
 |-- po_vendor_name: string (nullable = true)
 |-- id: long (nullab

In [0]:
#TYPE CASTING and Create Silver


silver_df = purchase_orders \
.withColumn("quantity", F.col("quantity").cast("int")) \
.withColumn("base_amount", F.col("base_amount").cast("double")) \
.withColumn("cgst_rate", F.col("cgst_rate").cast("double")) \
.withColumn("sgst_rate", F.col("sgst_rate").cast("double")) \
.withColumn("igst_rate", F.col("igst_rate").cast("double")) \
.withColumn("cgst_amt", F.col("cgst_amt").cast("double")) \
.withColumn("sgst_amt", F.col("sgst_amt").cast("double")) \
.withColumn("igst_amt", F.col("igst_amt").cast("double")) \
.withColumn("cess_amt", F.col("cess_amt").cast("double")) \
.withColumn("total_amount", F.col("total_amount").cast("double"))
silver_df.printSchema()

root
 |-- po_id: string (nullable = true)
 |-- po_date: date (nullable = true)
 |-- buyer_gstin: string (nullable = true)
 |-- buyer_state: string (nullable = true)
 |-- hsn_code: string (nullable = true)
 |-- product_desc: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit: string (nullable = true)
 |-- base_amount: double (nullable = true)
 |-- cgst_rate: double (nullable = true)
 |-- sgst_rate: double (nullable = true)
 |-- igst_rate: double (nullable = true)
 |-- cgst_amt: double (nullable = true)
 |-- sgst_amt: double (nullable = true)
 |-- igst_amt: double (nullable = true)
 |-- cess_amt: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- ewb_number: string (nullable = true)
 |-- ewb_generated_date: string (nullable = true)
 |-- place_of_supply: string (nullable = true)
 |-- delivery_state: string (nullable = true)
 |-- invoice_billing_name: string (nullable = true)
 |-- po_vendor_name: string (nullable = true)
 |-- id: long (nullab

###Vendor Join

In [0]:

#Renamed coz Ambigous
vendors = vendors \
    .withColumnRenamed("status","vendor_status") \
    .withColumnRenamed("id","vendor_row_id")

#JOIN Vendor master
silver_df = silver_df.join(
    vendors,
    on="vendor_id",
    how="left"
)

In [0]:
#CHECK JOIN

silver_df.select(
    "vendor_id",
    "trade_name",
    "legal_name",
    "status",
    "composition_flag"
).show(5, False)

+---------+-----------------------------+-----------------------------+------+----------------+
|vendor_id|trade_name                   |legal_name                   |status|composition_flag|
+---------+-----------------------------+-----------------------------+------+----------------+
|V00605   |Lakshmi Solutions Enterprises|Lakshmi Solutions Enterprises|NULL  |0               |
|V00765   |Lakshmi Solutions Enterprises|Lakshmi Solutions Enterprises|NULL  |0               |
|V00543   |Standard Pharma Pvt Ltd      |Standard Pharma Pvt Ltd      |NULL  |0               |
|V00909   |National Trading Trading Co  |National Trading Trading Co  |NULL  |0               |
|V00279   |Apex Plastics Pvt Ltd        |Apex Plastics Pvt Ltd        |NULL  |0               |
+---------+-----------------------------+-----------------------------+------+----------------+
only showing top 5 rows


###Invoice Join

In [0]:
vendor_invoices = vendor_invoices.dropDuplicates(["po_id"])


from pyspark.sql import functions as F

vendor_invoices.groupBy("po_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(20, False)

+-----+-----+
|po_id|count|
+-----+-----+
+-----+-----+



In [0]:
#JOIN Vendor Invoices

vendors_invoices = vendor_invoices.drop("ingestion_timestamp")

silver_df = silver_df.join(
    vendor_invoices,
    on="po_id",
    how="left"
)

In [0]:
#CHECK Join
silver_df.select(
    "po_id",
    "invoice_number",
    "invoice_date",
    "gstr2b_itc_available"
).show(5, False)

+---------+--------------+------------+--------------------+
|po_id    |invoice_number|invoice_date|gstr2b_itc_available|
+---------+--------------+------------+--------------------+
|PO0049982|INV00049982   |26/04/2023  |806342.2            |
|PO0049960|INV00049960   |20/04/2023  |212305.99           |
|PO0049948|INV00049948   |02/04/2023  |596762.76           |
|PO0049933|INV00049933   |06/04/2023  |400113.37           |
|PO0049931|INV00049931   |06/04/2023  |770102.75           |
+---------+--------------+------------+--------------------+
only showing top 5 rows


###GST VALID

In [0]:
#GSTIN Format Validation

GST_REGEX = r'^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][A-Z0-9]Z[A-Z0-9]$'

#JOIN GSTIN

silver_df = silver_df.withColumn(
    "is_gstin_valid",
    F.col("buyer_gstin").rlike(GST_REGEX)
)

In [0]:
silver_df.select(
    "buyer_gstin",
    "is_gstin_valid"
).show(10, False)

+---------------+--------------+
|buyer_gstin    |is_gstin_valid|
+---------------+--------------+
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
|27IIIII3249D1ZL|true          |
+---------------+--------------+
only showing top 10 rows


In [0]:
#CHECK

silver_df.filter(
    F.col("is_gstin_valid") == False
).count()

0

###GST State name

In [0]:
#JOIN GST NAME
silver_df = silver_df.withColumn(
    "gst_state_code",
    F.substring("buyer_gstin", 1, 2)
)

In [0]:
silver_df.select(
    "buyer_gstin",
    "gst_state_code"
).show(10, False)

+---------------+--------------+
|buyer_gstin    |gst_state_code|
+---------------+--------------+
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
|27IIIII3249D1ZL|27            |
+---------------+--------------+
only showing top 10 rows


###State Mismatch

In [0]:
#Check Distinct State Codes

silver_df.select(
    "gst_state_code"
).distinct().show()


+--------------+
|gst_state_code|
+--------------+
|            27|
+--------------+



In [0]:


from itertools import chain
from pyspark.sql import functions as F

state_map = {
    "01": "JAMMU AND KASHMIR",
    "02": "HIMACHAL PRADESH",
    "03": "PUNJAB",
    "04": "CHANDIGARH",
    "05": "UTTARAKHAND",
    "06": "HARYANA",
    "07": "DELHI",
    "08": "RAJASTHAN",
    "09": "UTTAR PRADESH",
    "10": "BIHAR",
    "11": "SIKKIM",
    "12": "ARUNACHAL PRADESH",
    "13": "NAGALAND",
    "14": "MANIPUR",
    "15": "MIZORAM",
    "16": "TRIPURA",
    "17": "MEGHALAYA",
    "18": "ASSAM",
    "19": "WEST BENGAL",
    "20": "JHARKHAND",
    "21": "ODISHA",
    "22": "CHHATTISGARH",
    "23": "MADHYA PRADESH",
    "24": "GUJARAT",
    "25": "DAMAN AND DIU",
    "26": "DADRA AND NAGAR HAVELI",
    "27": "MAHARASHTRA",
    "28": "ANDHRA PRADESH",
    "29": "KARNATAKA",
    "30": "GOA",
    "31": "LAKSHADWEEP ISLANDS",
    "32": "KERALA",
    "33": "TAMIL NADU",
    "34": "PONDICHERRY",
    "35": "ANDAMAN AND NICOBAR ISLANDS",
    "36": "TELANGANA",
    "37": "ANDHRA PRADESH (NEW)"
}

mapping_expr = F.create_map(
    [F.lit(x) for x in chain(*state_map.items())]
)

silver_df = silver_df.withColumn(
    "gst_state_name",
    mapping_expr[F.col("gst_state_code")]
)

In [0]:
silver_df.select(
    "gst_state_code",
    "gst_state_name"
).distinct().show(10, False)

+--------------+--------------+
|gst_state_code|gst_state_name|
+--------------+--------------+
|27            |MAHARASHTRA   |
+--------------+--------------+



In [0]:
silver_df.select("buyer_state").distinct().show(truncate=False)

+-----------+
|buyer_state|
+-----------+
|Maharashtra|
|Karnataka  |
+-----------+



In [0]:
silver_df = silver_df.withColumn(
    "state_mismatch",
    F.when(
        F.lower(F.col("buyer_state")) !=
        F.lower(F.col("gst_state_name")),
        1
    ).otherwise(0)
)

###Blacklist Join

In [0]:
#JOin BlackList

blacklist_df = cbic_blacklist.withColumnRenamed(
    "gstin",
    "blacklist_gstin"
)
silver_df = silver_df.join(
    blacklist_df.select("blacklist_gstin"),
    silver_df.gstin == blacklist_df.blacklist_gstin,
    "left"
)


silver_df = silver_df.withColumn(
    "blacklisted",
    F.when(
        F.col("blacklist_gstin").isNotNull(),
        1
    ).otherwise(0)
)

In [0]:
#CHECK JOIN


silver_df.groupBy(
    "blacklisted"
).count().show()

+-----------+-----+
|blacklisted|count|
+-----------+-----+
|          1| 2549|
|          0|47931|
+-----------+-----+



###HSN JOIN

In [0]:
#JOIN HSN RATE



hsn_rate_schedule = hsn_rate_schedule.withColumn(
    "gst_rate",
    F.col("gst_rate").cast("double")
)


silver_df = silver_df.join(
    hsn_rate_schedule.select(
        "hsn_code",
        "gst_rate"
    ),
    on="hsn_code",
    how="left"
)

In [0]:
#CHECK JOIN


silver_df.select(
    "hsn_code",
    "gst_rate"
).show(10, False)

+--------+--------+
|hsn_code|gst_rate|
+--------+--------+
|29051100|18.0    |
|48026290|12.0    |
|72101200|18.0    |
|28182010|18.0    |
|85423100|18.0    |
|84713020|18.0    |
|39269099|18.0    |
|27101290|18.0    |
|29053100|18.0    |
|38200000|18.0    |
+--------+--------+
only showing top 10 rows


In [0]:
silver_df.filter(
    F.col("gst_rate").isNull()
).count()

0

In [0]:
#CALCULATE ACTUAL GST RATE

silver_df = silver_df.withColumn(
    "actual_gst_rate",
    F.col("cgst_rate") +
    F.col("sgst_rate") +
    F.col("igst_rate")
)


silver_df.select(
    "cgst_rate",
    "sgst_rate",
    "igst_rate",
    "actual_gst_rate"
).show(10, False)

+---------+---------+---------+---------------+
|cgst_rate|sgst_rate|igst_rate|actual_gst_rate|
+---------+---------+---------+---------------+
|0.0      |0.0      |18.0     |18.0           |
|0.0      |0.0      |12.0     |12.0           |
|0.0      |0.0      |18.0     |18.0           |
|0.0      |0.0      |18.0     |18.0           |
|0.0      |0.0      |12.0     |12.0           |
|0.0      |0.0      |18.0     |18.0           |
|0.0      |0.0      |18.0     |18.0           |
|9.0      |9.0      |0.0      |18.0           |
|0.0      |0.0      |18.0     |18.0           |
|0.0      |0.0      |18.0     |18.0           |
+---------+---------+---------+---------------+
only showing top 10 rows


In [0]:
#TAX-001 (HSN Rate Mismatch Rule)


silver_df = silver_df.withColumn(
    "rate_mismatch",
    F.when(
        F.col("actual_gst_rate") != F.col("gst_rate"),
        1
    ).otherwise(0)
)


silver_df.groupBy(
    "rate_mismatch"
).count().show()

+-------------+-----+
|rate_mismatch|count|
+-------------+-----+
|            1| 3101|
|            0|47379|
+-------------+-----+



In [0]:
silver_df.filter(
    F.col("rate_mismatch") == 1
).select(
    "po_id",
    "hsn_code",
    "gst_rate",
    "actual_gst_rate"
).show(10, False)

+---------+--------+--------+---------------+
|po_id    |hsn_code|gst_rate|actual_gst_rate|
+---------+--------+--------+---------------+
|PO0049931|85423100|18.0    |12.0           |
|PO0049659|85423100|18.0    |28.0           |
|PO0049651|72101200|18.0    |5.0            |
|PO0049614|84314900|18.0    |5.0            |
|PO0049607|90185090|12.0    |28.0           |
|PO0049400|40169990|18.0    |12.0           |
|PO0049376|38200000|18.0    |28.0           |
|PO0048931|85423100|18.0    |12.0           |
|PO0048659|85423100|18.0    |28.0           |
|PO0048651|72101200|18.0    |5.0            |
+---------+--------+--------+---------------+
only showing top 10 rows


###Historical Join

In [0]:
# Historical PO YEAR_MONTH

historical_po_values.select(
    "year_month"
).distinct().show(20, False)

silver_df = silver_df.withColumn(
    "year_month",
    F.date_format(
        F.col("po_date"),
        "yyyy-MM"
    )
)

silver_df.select(
    "po_date",
    "year_month"
).show(10, False)

+----------+
|year_month|
+----------+
|2024-01   |
|2023-10   |
|2014-07   |
|2024-02   |
|2023-06   |
|2012-11   |
|2020-12   |
|2020-10   |
|2016-12   |
|2018-10   |
|2022-05   |
|2023-01   |
|2011-12   |
|2017-04   |
|2013-12   |
|2016-08   |
|2012-10   |
|2023-03   |
|2021-04   |
|2023-08   |
+----------+
only showing top 20 rows
+----------+----------+
|po_date   |year_month|
+----------+----------+
|2023-04-23|2023-04   |
|2023-04-18|2023-04   |
|2023-04-02|2023-04   |
|2023-04-04|2023-04   |
|2023-04-05|2023-04   |
|2023-04-18|2023-04   |
|2023-04-29|2023-04   |
|2023-04-22|2023-04   |
|2023-04-15|2023-04   |
|2023-04-22|2023-04   |
+----------+----------+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

historical_po_values = (
    historical_po_values
    .groupBy(
        "vendor_id",
        "year_month"
    )
    .agg(
        F.avg("total_po_value").alias("total_po_value"),
        F.avg("po_count").alias("po_count")
    )
)

# Validation
historical_po_values.groupBy(
    "vendor_id",
    "year_month"
).count().filter(
    F.col("count") > 1
).show()

+---------+----------+-----+
|vendor_id|year_month|count|
+---------+----------+-----+
+---------+----------+-----+



In [0]:
# JOIN HISTORICAL DATA
# ==========================================

silver_df = silver_df.join(
    historical_po_values,
    on=["vendor_id", "year_month"],
    how="left"
)

# Average PO Value
silver_df = silver_df.withColumn(
    "avg_po_value",
    F.col("total_po_value") / F.col("po_count")
)

# Spike Ratio
silver_df = silver_df.withColumn(
    "spike_ratio",
    F.round(
        F.col("total_amount") /
        F.col("avg_po_value"),
        2
    )
)

# Value Spike Flag
silver_df = silver_df.withColumn(
    "value_spike",
    F.when(
        F.col("spike_ratio") > 3,
        1
    ).otherwise(0)
)

In [0]:
#CHeck Join

silver_df.select(
    "vendor_id",
    "year_month",
    "total_po_value",
    "po_count"
).show(10, False)

+---------+----------+------------------+--------+
|vendor_id|year_month|total_po_value    |po_count|
+---------+----------+------------------+--------+
|V00605   |2023-04   |921533.91         |1.0     |
|V00765   |2023-04   |3084950.65        |1.0     |
|V00543   |2023-04   |NULL              |NULL    |
|V00909   |2023-04   |NULL              |NULL    |
|V00279   |2023-04   |2604251.4233333333|2.0     |
|V00831   |2023-04   |NULL              |NULL    |
|V00342   |2023-04   |NULL              |NULL    |
|V00031   |2023-04   |NULL              |NULL    |
|V00337   |2023-04   |1998038.9100000001|1.0     |
|V00053   |2023-04   |NULL              |NULL    |
+---------+----------+------------------+--------+
only showing top 10 rows


In [0]:
silver_df.filter(
    F.col("total_po_value").isNull()
).count()

23255

In [0]:
#Calculate Average PO Value

#avg_po_value = total_po_value / po_count


silver_df = silver_df.withColumn(
    "avg_po_value",
    F.round(
        F.col("total_po_value") / F.col("po_count"),
        2
    )
)
silver_df.select(
    "vendor_id",
    "total_po_value",
    "po_count",
    "avg_po_value"
).show(10, False)

+---------+------------------+--------+------------+
|vendor_id|total_po_value    |po_count|avg_po_value|
+---------+------------------+--------+------------+
|V00605   |921533.91         |1.0     |921533.91   |
|V00765   |3084950.65        |1.0     |3084950.65  |
|V00543   |NULL              |NULL    |NULL        |
|V00909   |NULL              |NULL    |NULL        |
|V00279   |2604251.4233333333|2.0     |1302125.71  |
|V00831   |NULL              |NULL    |NULL        |
|V00342   |NULL              |NULL    |NULL        |
|V00031   |NULL              |NULL    |NULL        |
|V00337   |1998038.9100000001|1.0     |1998038.91  |
|V00053   |NULL              |NULL    |NULL        |
+---------+------------------+--------+------------+
only showing top 10 rows


In [0]:
#Calculate Spike Ratio

#spike_ratio =current PO amount / average historical PO value

silver_df = silver_df.withColumn(
    "spike_ratio",
    F.round(
        F.col("total_amount") / F.col("avg_po_value"),
        2
    )
)


silver_df.select(
    "total_amount",
    "avg_po_value",
    "spike_ratio"
).show(10, False)

+------------+------------+-----------+
|total_amount|avg_po_value|spike_ratio|
+------------+------------+-----------+
|5286021.09  |921533.91   |5.74       |
|1981522.6   |3084950.65  |0.64       |
|3912111.41  |NULL        |NULL       |
|2622965.41  |NULL        |NULL       |
|4791750.47  |1302125.71  |3.68       |
|5078853.16  |NULL        |NULL       |
|1307688.9   |NULL        |NULL       |
|2056112.99  |NULL        |NULL       |
|664168.63   |1998038.91  |0.33       |
|2859141.56  |NULL        |NULL       |
+------------+------------+-----------+
only showing top 10 rows


In [0]:
#FRD-004 Value Spike Detection

#Invoice Value Spike
#Current Value > 3 × Historical Average


silver_df = silver_df.withColumn(
    "value_spike",
    F.when(
        F.col("spike_ratio") > 3.0,
        1
    ).otherwise(0)
)

silver_df.groupBy(
    "value_spike"
).count().show()


silver_df.filter(
    F.col("value_spike") == 1
).select(
    "po_id",
    "vendor_id",
    "total_amount",
    "avg_po_value",
    "spike_ratio"
).show(10, False)

+-----------+-----+
|value_spike|count|
+-----------+-----+
|          1| 7108|
|          0|43372|
+-----------+-----+

+---------+---------+------------+------------+-----------+
|po_id    |vendor_id|total_amount|avg_po_value|spike_ratio|
+---------+---------+------------+------------+-----------+
|PO0049982|V00605   |5286021.09  |921533.91   |5.74       |
|PO0049931|V00279   |4791750.47  |1302125.71  |3.68       |
|PO0049807|V00243   |3200235.43  |576427.34   |5.55       |
|PO0049651|V00208   |4473325.29  |1439930.68  |3.11       |
|PO0049594|V00079   |5734959.65  |438529.32   |13.08      |
|PO0049578|V00133   |4321831.34  |1328006.22  |3.25       |
|PO0049397|V00857   |3022884.22  |217208.82   |13.92      |
|PO0049085|V00568   |4822907.85  |1469435.47  |3.28       |
|PO0048840|V00097   |4295189.08  |1007839.01  |4.26       |
|PO0048797|V00731   |5576127.92  |1440633.54  |3.87       |
+---------+---------+------------+------------+-----------+
only showing top 10 rows


## SIM_001 NAME MISMATCH

In [0]:
#Name Similarity (SIM-001)

#invoice_billing_name vs trade_name


silver_df.select(
    "invoice_billing_name",
    "trade_name"
).show(10, False)

+------------------------------+---------------------------------+
|invoice_billing_name          |trade_name                       |
+------------------------------+---------------------------------+
|Infra Electronics Pvt Ltd     |Lakshmi Solutions Enterprises    |
|detimiL ted                   |Lakshmi Solutions Enterprises    |
|Northern Solutions Corporation|Standard Pharma Pvt Ltd          |
|Krishna Textiles Manufacturing|National Trading Trading Co      |
|Metro Textiles Traders        |Apex Plastics Pvt Ltd            |
|Indian Metals Enterprises     |Capital Components Traders       |
|Capital Steel Works           |Eastern Engineering Works        |
|Shree Pharma Enterprises      |Capital Components Traders       |
|Power Logistics Trading Co    |Apex Textiles Industries         |
|Lakshmi Systems Ltd           |Shree Electronics Private Limited|
+------------------------------+---------------------------------+
only showing top 10 rows


In [0]:
%pip install rapidfuzz

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from rapidfuzz import fuzz
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

In [0]:
def similarity_score(name1, name2):
    if name1 is None or name2 is None:
        return 0.0

    return fuzz.ratio(
        str(name1).lower(),
        str(name2).lower()
    ) / 100.0

similarity_udf = udf(
    similarity_score,
    DoubleType()
)

In [0]:
silver_df = silver_df.withColumn(
    "name_sim_score",
    similarity_udf(
        F.col("invoice_billing_name"),
        F.col("trade_name")
    )
)

silver_df.select(
    "invoice_billing_name",
    "trade_name",
    "name_sim_score"
).show(10, False)

+------------------------------+---------------------------------+-------------------+
|invoice_billing_name          |trade_name                       |name_sim_score     |
+------------------------------+---------------------------------+-------------------+
|Infra Electronics Pvt Ltd     |Lakshmi Solutions Enterprises    |0.33333333333333337|
|detimiL ted                   |Lakshmi Solutions Enterprises    |0.30000000000000004|
|Northern Solutions Corporation|Standard Pharma Pvt Ltd          |0.30188679245283023|
|Krishna Textiles Manufacturing|National Trading Trading Co      |0.45614035087719296|
|Metro Textiles Traders        |Apex Plastics Pvt Ltd            |0.37209302325581395|
|Indian Metals Enterprises     |Capital Components Traders       |0.4705882352941176 |
|Capital Steel Works           |Eastern Engineering Works        |0.5                |
|Shree Pharma Enterprises      |Capital Components Traders       |0.36               |
|Power Logistics Trading Co    |Apex Textil

In [0]:
silver_df.select(
    "invoice_billing_name",
    "trade_name",
    "name_sim_score"
).orderBy(
    F.col("name_sim_score").asc()
).show(20, False)

+--------------------+---------------------------+-------------------+
|invoice_billing_name|trade_name                 |name_sim_score     |
+--------------------+---------------------------+-------------------+
|gnirutcaing         |Lakshmi Systems Ltd        |0.1333333333333333 |
|oC gnida Co         |Lakshmi Systems Ltd        |0.1333333333333333 |
|gnirutcaing         |Lakshmi Systems Ltd        |0.1333333333333333 |
|gnirutcaing         |Lakshmi Systems Ltd        |0.1333333333333333 |
|oC gnida Co         |Lakshmi Systems Ltd        |0.1333333333333333 |
|dtL amraLtd         |Premier Systems Enterprises|0.1578947368421053 |
|Modern Systems Works|Capital Pharma Ltd         |0.1578947368421053 |
|dtL amraLtd         |Premier Systems Enterprises|0.1578947368421053 |
|dtL tvP Ltd         |Eastern Engineering Works  |0.16666666666666663|
|gnirutcaing         |Shree Pharma Enterprises   |0.17142857142857137|
|oC gnida Co         |Shree Pharma Enterprises   |0.17142857142857137|
|oC gn

In [0]:
#Mismatch Flag


silver_df = silver_df.withColumn(
    "name_mismatch",
    F.when(
        F.col("name_sim_score") < 0.85,
        1
    ).otherwise(0)
)


silver_df.groupBy(
    "name_mismatch"
).count().show()

+-------------+-----+
|name_mismatch|count|
+-------------+-----+
|            1|49784|
|            0|  696|
+-------------+-----+



In [0]:
silver_df.filter(
    F.col("name_mismatch") == 1
).select(
    "invoice_billing_name",
    "trade_name",
    "name_sim_score"
).show(20, False)

+-----------------------------------+----------------------------------+-------------------+
|invoice_billing_name               |trade_name                        |name_sim_score     |
+-----------------------------------+----------------------------------+-------------------+
|Infra Electronics Pvt Ltd          |Lakshmi Solutions Enterprises     |0.33333333333333337|
|detimiL ted                        |Lakshmi Solutions Enterprises     |0.30000000000000004|
|Northern Solutions Corporation     |Standard Pharma Pvt Ltd           |0.30188679245283023|
|Krishna Textiles Manufacturing     |National Trading Trading Co       |0.45614035087719296|
|Metro Textiles Traders             |Apex Plastics Pvt Ltd             |0.37209302325581395|
|Indian Metals Enterprises          |Capital Components Traders        |0.4705882352941176 |
|Capital Steel Works                |Eastern Engineering Works         |0.5                |
|Shree Pharma Enterprises           |Capital Components Traders       

###Duplicate Invoice Detection (DQ-001)

In [0]:
#No duplicate invoice_number
#for same vendor GSTIN
#in same month



from pyspark.sql.window import Window

dup_window = Window.partitionBy(
    "gstin",
    "invoice_number",
    "year",
    "month"
)

silver_df = silver_df.withColumn(
    "invoice_dup_count",
    F.count("*").over(dup_window)
)


#Flag

silver_df = silver_df.withColumn(
    "duplicate_invoice",
    F.when(
        F.col("invoice_dup_count") > 1,
        1
    ).otherwise(0)
)

#CHECK
silver_df.groupBy(
    "duplicate_invoice"
).count().show()

+-----------------+-----+
|duplicate_invoice|count|
+-----------------+-----+
|                0|50000|
|                1|  480|
+-----------------+-----+



###Director overlap Flag

In [0]:
director_counts = (
    silver_df
    .groupBy("director_pan_1")
    .agg(
        F.countDistinct("vendor_id")
        .alias("director_vendor_count")
    )
)

silver_df = silver_df.join(
    director_counts,
    "director_pan_1",
    "left"
)

silver_df = silver_df.withColumn(
    "director_overlap_flag",
    F.when(
        F.col("director_vendor_count") > 1,
        1
    ).otherwise(0)
)

###Invoice Timing

In [0]:

silver_df = silver_df.withColumn(
    "invoice_date",
    F.coalesce(
        F.to_date("invoice_date", "dd/MM/yyyy"),
        F.to_date("invoice_date", "yyyy-MM-dd")
    )
)

silver_df = silver_df.withColumn(
    "late_invoice",
    F.when(
        F.datediff(
            F.col("invoice_date"),
            F.col("po_date")
        ) > 30,
        1
    ).otherwise(0)
)

###filing_compliance_rate

In [0]:
silver_df = silver_df.withColumn(
    "filing_compliance_rate",
    (
        F.col("filing_history_q1") +
        F.col("filing_history_q2") +
        F.col("filing_history_q3") +
        F.col("filing_history_q4") +
        F.col("filing_history_q5") +
        F.col("filing_history_q6")
    ) / 6
)

In [0]:
silver_df = silver_df.withColumn(
    "vendor_non_filer",
    F.when(
        (
            F.col("filing_history_q1") +
            F.col("filing_history_q2") +
            F.col("filing_history_q3") +
            F.col("filing_history_q4") +
            F.col("filing_history_q5") +
            F.col("filing_history_q6")
        ) < 4,
        1
    ).otherwise(0)
)

###

##EWB-001 Missing E-Way Bill

In [0]:
# If total_amount > 50000
# then ewb_number must exist

silver_df = silver_df.withColumn(
    "missing_ewb",
    F.when(
        (F.col("total_amount") > 50000) &
        (F.col("ewb_number").isNull()),
        1
    ).otherwise(0)
)


silver_df.groupBy(
    "missing_ewb"
).count().show()

+-----------+-----+
|missing_ewb|count|
+-----------+-----+
|          1| 5620|
|          0|44860|
+-----------+-----+



In [0]:
#Composite Rule Score


silver_df = silver_df.withColumn(
    "rule_fail_count",
    F.col("blacklisted") +
    F.col("rate_mismatch") +
    F.col("value_spike") +
    F.col("name_mismatch") +
    F.col("duplicate_invoice") +
    F.col("missing_ewb")+
    F.col("director_overlap_flag")+
    F.col("state_mismatch")+
    F.col("late_invoice")+
    F.col("vendor_non_filer")
)



silver_df = silver_df.withColumn(
    "rule_score",
    F.round(
        (1 - (F.col("rule_fail_count") / 10)) * 100,
        2
    )
)



silver_df.select(
    "rule_fail_count",
    "rule_score"
).show(20, False)

+---------------+----------+
|rule_fail_count|rule_score|
+---------------+----------+
|4              |60.0      |
|4              |60.0      |
|4              |60.0      |
|6              |40.0      |
|4              |60.0      |
|4              |60.0      |
|4              |60.0      |
|6              |40.0      |
|3              |70.0      |
|5              |50.0      |
|4              |60.0      |
|4              |60.0      |
|5              |50.0      |
|4              |60.0      |
|4              |60.0      |
|5              |50.0      |
|4              |60.0      |
|4              |60.0      |
|5              |50.0      |
|4              |60.0      |
+---------------+----------+
only showing top 20 rows


In [0]:
silver_df.groupBy(
    "rule_score"
).count().orderBy("rule_score").show(20)

+----------+-----+
|rule_score|count|
+----------+-----+
|      40.0|   44|
|      50.0| 1075|
|      60.0|10236|
|      70.0|26382|
|      80.0|12557|
|      90.0|  186|
+----------+-----+



In [0]:
silver_df.select(
    "registration_date",
    "invoice_date",
    "ewb_generated_date"
).show(20, False)

+-----------------+------------+------------------+
|registration_date|invoice_date|ewb_generated_date|
+-----------------+------------+------------------+
|2022-01-20       |2023-04-26  |22/04/2023        |
|2022-01-20       |2023-04-20  |19/04/2023        |
|2026-05-18       |2023-04-02  |03/04/2023        |
|2022-07-06       |2023-04-06  |04/04/2023        |
|2019-11-26       |2023-04-06  |06/04/2023        |
|2019-01-20       |2023-04-20  |18/04/2023        |
|2026-05-18       |2023-04-30  |29/04/2023        |
|2019-01-20       |2023-04-24  |22/04/2023        |
|2017-05-13       |2023-04-18  |16/04/2023        |
|2014-07-23       |2023-04-22  |22/04/2023        |
|2022-06-03       |2023-04-08  |06/04/2023        |
|2022-06-03       |2023-04-07  |04/04/2023        |
|2014-09-25       |2023-04-23  |22/04/2023        |
|2014-10-30       |2023-04-13  |NULL              |
|2022-11-20       |2023-04-13  |14/04/2023        |
|2022-11-20       |2023-04-16  |15/04/2023        |
|2020-12-10 

In [0]:
#Date Format change

silver_df = silver_df.withColumn(
    "registration_date",
    F.to_date(
        F.col("registration_date"),
        "dd/MM/yyyy"
    )
)

silver_df = silver_df.withColumn(
    "invoice_date",
    F.to_date(
        F.col("invoice_date"),
        "dd/MM/yyyy"
    )
)

silver_df = silver_df.withColumn(
    "ewb_generated_date",
    F.to_date(
        F.col("ewb_generated_date"),
        "dd/MM/yyyy"
    )
)

In [0]:
silver_df.select(
    "registration_date",
    "invoice_date",
    "ewb_generated_date"
).show(10, False)

silver_df.printSchema()

+-----------------+------------+------------------+
|registration_date|invoice_date|ewb_generated_date|
+-----------------+------------+------------------+
|2022-01-20       |2023-04-26  |2023-04-22        |
|2022-01-20       |2023-04-20  |2023-04-19        |
|2026-05-18       |2023-04-02  |2023-04-03        |
|2022-07-06       |2023-04-06  |2023-04-04        |
|2019-11-26       |2023-04-06  |2023-04-06        |
|2019-01-20       |2023-04-20  |2023-04-18        |
|2026-05-18       |2023-04-30  |2023-04-29        |
|2019-01-20       |2023-04-24  |2023-04-22        |
|2017-05-13       |2023-04-18  |2023-04-16        |
|2014-07-23       |2023-04-22  |2023-04-22        |
+-----------------+------------+------------------+
only showing top 10 rows
root
 |-- director_pan_1: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- hsn_code: string (nullable = true)
 |-- po_id: string (nullable = true)
 |-- po_date: date (nullable = true)

In [0]:
er_df = silver_df \
.withColumn(
    "year",
    F.col("year").cast("int")
) \
.withColumn(
    "month",
    F.col("month").cast("int")
)

##GROUND TRUTH

In [0]:
#change schema

ground_truth = ground_truth.withColumn(
    "is_anomalous",
    F.col("is_anomalous").cast("int")
)

display(ground_truth.limit(10))

po_id,anomaly_code,is_anomalous,severity
PO0000001,CLEAN,0,Low
PO0000002,CLEAN,0,Low
PO0000003,CLEAN,0,Low
PO0000004,CLEAN,0,Low
PO0000005,GST-001,1,High
PO0000006,CLEAN,0,Low
PO0000007,CLEAN,0,Low
PO0000008,CLEAN,0,Low
PO0000009,CLEAN,0,Low
PO0000010,CLEAN,0,Low


In [0]:
#Join ground

silver_df = silver_df.join(
    ground_truth,
    on="po_id",
    how="left"
)
print("JOIN SUCCESS")

JOIN SUCCESS


In [0]:
ground_truth.groupBy("severity").count().show()

+--------+-----+
|severity|count|
+--------+-----+
|    High| 3073|
|     Low|47407|
+--------+-----+



In [0]:
ground_truth.filter(
    F.col("severity") == "Critical"
).show(20, False)

+-----+------------+------------+--------+
|po_id|anomaly_code|is_anomalous|severity|
+-----+------------+------------+--------+
+-----+------------+------------+--------+



In [0]:
ground_truth.printSchema()
display(ground_truth.limit(5))

root
 |-- po_id: string (nullable = true)
 |-- anomaly_code: string (nullable = true)
 |-- is_anomalous: integer (nullable = true)
 |-- severity: string (nullable = true)



po_id,anomaly_code,is_anomalous,severity
PO0000001,CLEAN,0,Low
PO0000002,CLEAN,0,Low
PO0000003,CLEAN,0,Low
PO0000004,CLEAN,0,Low
PO0000005,GST-001,1,High


In [0]:
from pyspark.sql.functions import col, coalesce, lit

silver_df = (
    silver_df
    .withColumn(
        "is_anomalous",
        coalesce(col("is_anomalous"), lit(0))
    )
    .withColumn(
        "anomaly_code",
        coalesce(col("anomaly_code"), lit("CLEAN"))
    )
)

In [0]:
# Using Spark SQL to find null counts in each column
from collections import Counter

for col_name, count in Counter(silver_df.columns).items():
    if count > 1:
        print(col_name, count)



id 2
ingestion_timestamp 2


In [0]:
for i, c in enumerate(silver_df.columns):
    print(i, c)

0 po_id
1 director_pan_1
2 vendor_id
3 year_month
4 hsn_code
5 po_date
6 buyer_gstin
7 buyer_state
8 product_desc
9 quantity
10 unit
11 base_amount
12 cgst_rate
13 sgst_rate
14 igst_rate
15 cgst_amt
16 sgst_amt
17 igst_amt
18 cess_amt
19 total_amount
20 ewb_number
21 ewb_generated_date
22 place_of_supply
23 delivery_state
24 invoice_billing_name
25 po_vendor_name
26 id
27 status
28 ingestion_timestamp
29 year
30 month
31 gstin
32 trade_name
33 legal_name
34 billing_state
35 registration_date
36 vendor_status
37 composition_flag
38 filing_history_q1
39 filing_history_q2
40 filing_history_q3
41 filing_history_q4
42 filing_history_q5
43 filing_history_q6
44 director_pan_2
45 vendor_row_id
46 ingestion_timestamp
47 invoice_date
48 invoice_number
49 gstr2b_itc_available
50 itc_claimed_by_buyer
51 id
52 is_gstin_valid
53 gst_state_code
54 gst_state_name
55 state_mismatch
56 blacklist_gstin
57 blacklisted
58 gst_rate
59 actual_gst_rate
60 rate_mismatch
61 total_po_value
62 po_count
63 avg_po_

In [0]:
silver_df = silver_df.toDF(*[
    f"{col}_{i}" if silver_df.columns.count(col) > 1 else col
    for i, col in enumerate(silver_df.columns)
])

###vendor_age_days / is_marh_invoice / hsn_rate+delta

In [0]:
from pyspark.sql import functions as F

silver_df = (
    silver_df
    .withColumn(
        "vendor_age_days",
        F.datediff(F.current_date(), F.col("registration_date"))
    )
    .withColumn(
        "is_march_invoice",
        F.when(F.month("invoice_date") == 3, 1).otherwise(0)
    )
    .withColumn(
        "hsn_rate_delta",
        F.abs(F.col("actual_gst_rate") - F.col("gst_rate"))
    )
)

###Invoice count same Vendor same day

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("vendor_id", "invoice_date")

silver_df = silver_df.withColumn(
    "invoice_count_same_vendor_same_day",
    F.count("*").over(w)
)

###Vendor Po Frequency

In [0]:
vendor_window = Window.partitionBy("vendor_id")

silver_df = silver_df.withColumn(
    "vendor_po_frequency",
    F.count("po_id").over(vendor_window)
)

###High Value Invoice Flag

In [0]:
silver_df = silver_df.withColumn(
    "high_value_invoice_flag",
    F.when(F.col("total_amount") > 500000, 1).otherwise(0)
)

###Invoice To avg Ratio

In [0]:
silver_df = silver_df.withColumn(
    "invoice_to_avg_ratio",
    F.when(
        F.col("avg_po_value") > 0,
        F.col("total_amount") / F.col("avg_po_value")
    ).otherwise(0)
)

##SAVE

In [0]:
print("Total Columns :", len(silver_df.columns))
print("Unique Columns :", len(set(silver_df.columns)))

Total Columns : 88
Unique Columns : 88


In [0]:
display(silver_df.limit(5))

po_id,director_pan_1,vendor_id,year_month,hsn_code,po_date,buyer_gstin,buyer_state,product_desc,quantity,unit,base_amount,cgst_rate,sgst_rate,igst_rate,cgst_amt,sgst_amt,igst_amt,cess_amt,total_amount,ewb_number,ewb_generated_date,place_of_supply,delivery_state,invoice_billing_name,po_vendor_name,id_26,status,ingestion_timestamp_28,year,month,gstin,trade_name,legal_name,billing_state,registration_date,vendor_status,composition_flag,filing_history_q1,filing_history_q2,filing_history_q3,filing_history_q4,filing_history_q5,filing_history_q6,director_pan_2,vendor_row_id,ingestion_timestamp_46,invoice_date,invoice_number,gstr2b_itc_available,itc_claimed_by_buyer,id_51,is_gstin_valid,gst_state_code,gst_state_name,state_mismatch,blacklist_gstin,blacklisted,gst_rate,actual_gst_rate,rate_mismatch,total_po_value,po_count,avg_po_value,spike_ratio,value_spike,name_sim_score,name_mismatch,invoice_dup_count,duplicate_invoice,director_vendor_count,director_overlap_flag,late_invoice,filing_compliance_rate,vendor_non_filer,missing_ewb,rule_fail_count,rule_score,anomaly_code,is_anomalous,severity,vendor_age_days,is_march_invoice,hsn_rate_delta,invoice_count_same_vendor_same_day,vendor_po_frequency,high_value_invoice_flag,invoice_to_avg_ratio
PO0015591_A,AAAAA000000,V00001,2023-06,85414010,2023-06-09,27IIIII3249D1ZL,Karnataka,Solar panels,299,KG,30000.0,0.0,0.0,5.0,2700.0,2700.0,0.0,0.0,35400.0,EWB650964560,2023-06-10,Maharashtra,Maharashtra,Jindal Solutions Manufacturing,Jindal Solutions Manufacturing,44,null,2026-06-17T09:24:36.191Z,2023,6,23AAAAA6310001Y,Supreme Chemicals Manufacturing,Supreme Chemicals Manufacturing,Madhya Pradesh,2012-06-08,cancelled,0,0,1,1,0,1,0,BBBBB000000,0,2026-06-17T09:24:47.087Z,null,null,null,null,null,true,27,MAHARASHTRA,1,null,0,5.0,5.0,0,1152854.67,2.0,576427.34,0.06,0,0.5901639344262295,1,2,1,4,1,0,0.5,1,0,5,50.0,CLEAN,0,Low,5122,0,0.0,2,49,0,0.06141277060175529
PO0015591_B,AAAAA000000,V00001,2023-06,85414010,2023-06-09,27IIIII3249D1ZL,Karnataka,Solar panels,299,KG,30000.0,0.0,0.0,5.0,2700.0,2700.0,0.0,0.0,35400.0,EWB650964560,2023-06-10,Maharashtra,Maharashtra,Jindal Solutions Manufacturing,Jindal Solutions Manufacturing,44,null,2026-06-17T09:24:36.191Z,2023,6,23AAAAA6310001Y,Supreme Chemicals Manufacturing,Supreme Chemicals Manufacturing,Madhya Pradesh,2012-06-08,cancelled,0,0,1,1,0,1,0,BBBBB000000,0,2026-06-17T09:24:47.087Z,null,null,null,null,null,true,27,MAHARASHTRA,1,null,0,5.0,5.0,0,1152854.67,2.0,576427.34,0.06,0,0.5901639344262295,1,2,1,4,1,0,0.5,1,0,5,50.0,CLEAN,0,Low,5122,0,0.0,2,49,0,0.06141277060175529
PO0004722,AAAAA000000,V00001,2023-02,39201020,2023-02-28,27IIIII3249D1ZL,Karnataka,Plastic film,373,KG,3552495.24,0.0,0.0,18.0,0.0,0.0,639449.14,0.0,4191944.38,null,null,Maharashtra,Maharashtra,Eastern Engineering Works,Eastern Engineering Works,11,null,2026-06-17T09:24:36.191Z,2023,2,23AAAAA6310001Y,Supreme Chemicals Manufacturing,Supreme Chemicals Manufacturing,Madhya Pradesh,2012-06-08,cancelled,0,0,1,1,0,1,0,BBBBB000000,0,2026-06-17T09:24:47.087Z,2023-02-28,INV00004722,638248.0,639449.14,11,true,27,MAHARASHTRA,1,null,0,18.0,18.0,0,null,null,null,null,0,0.3571428571428571,1,1,0,4,1,0,0.5,1,1,5,50.0,GST-001,1,High,5122,0,0.0,1,49,1,0.0
PO0000005,AAAAA000000,V00001,2023-08,85423100,2023-08-23,27IIIII3249D1ZL,Karnataka,Semiconductors,5,MT,4883710.62,0.0,0.0,28.0,0.0,0.0,1367438.97,0.0,6251149.59,EWB849734394,2023-08-24,Maharashtra,Maharashtra,National Energy Corporation,National Energy Corporation,4,null,2026-06-17T09:24:36.191Z,2023,8,23AAAAA6310001Y,Supreme Chemicals Manufacturing,Supreme Chemicals Manufacturing,Madhya Pradesh,2012-06-08,cancelled,0,0,1,1,0,1,0,BBBBB000000,0,2026-06-17T09:24:47.087Z,2023-08-26,INV00000005,879067.91,1367438.97,4,true,27,MAHARASHTRA,1,null,0,18.0,28.0,1,1053848.43,1.0,1053848.43,5.93,1,0.31034482758620685,1,1,0,4,1,0,0.5,1,0,6,40.0,GST-001,1,High,5122,0,10.0,2,49,1,5.931734974449789
PO0043874,AAAAA000000,V00001,2023-08,84713020,2023-08-23,27IIIII3249D1ZL,Karnataka,Lap

In [0]:
silver_df.printSchema()

root
 |-- po_id: string (nullable = true)
 |-- director_pan_1: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- hsn_code: string (nullable = true)
 |-- po_date: date (nullable = true)
 |-- buyer_gstin: string (nullable = true)
 |-- buyer_state: string (nullable = true)
 |-- product_desc: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit: string (nullable = true)
 |-- base_amount: double (nullable = true)
 |-- cgst_rate: double (nullable = true)
 |-- sgst_rate: double (nullable = true)
 |-- igst_rate: double (nullable = true)
 |-- cgst_amt: double (nullable = true)
 |-- sgst_amt: double (nullable = true)
 |-- igst_amt: double (nullable = true)
 |-- cess_amt: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- ewb_number: string (nullable = true)
 |-- ewb_generated_date: date (nullable = true)
 |-- place_of_supply: string (nullable = true)
 |-- delivery_state: string (nullable 

In [0]:
silver_df.write \
.mode("overwrite") \
.parquet(
"/Volumes/workspace/default/gst-fraud&anomaly_detection/Silver/silver_purchase_orders"
)

In [0]:
saved_silver = spark.read.parquet(
"/Volumes/workspace/default/gst-fraud&anomaly_detection/Silver/silver_purchase_orders"
)

print(saved_silver.count())

50480
